# Quantum Pump BO Experiment - Hardware v5

## Five-Phase Integrated Workflow

### Rule No.1
- Any voltage change must be **step-wise** with maximum step `< 5 mV`
- In this notebook, we enforce `MAX_VOLTAGE_STEP_V = 4 mV`

### Phase 1
- Find pinch-off voltages for `V_ENT`, `V_P`, `V_EXIT` at gain `1e-7 A/V`

### Phase 2
- Find region where `0 < I < 4 nA` using 2D scan (`V_ENT`, `V_EXIT`) with fixed `V_P`

### Phase 3
- Experimenter reviews Phase 2 result
- `y`: manually switch amplifier dial to `1e-9 A/V`, then continue
- `n`: stop and save summary
- `r`: re-try Phase 2

### Phase 4
- Find `n = 1` region using constrained BO

### Phase 5
- Experimenter reviews Phase 4 result
- If `yes`, run pump-map plotting (6-panel)

In [ ]:
# Cell 1: Imports
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LogNorm
from pathlib import Path
from datetime import datetime
import warnings
import pandas as pd
import time
warnings.filterwarnings('ignore')

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from scipy.stats import norm, qmc
from scipy.optimize import minimize
from scipy.ndimage import label

# Hardware control
try:
    import pyvisa
    PYVISA_AVAILABLE = True
    print('✅ PyVISA imported successfully')
except ImportError:
    PYVISA_AVAILABLE = False
    print('⚠️ PyVISA not available - Simulation mode only')

print('='*60)
print('Quantum Pump BO Experiment - Hardware v5')
print('Five-Phase Workflow: Pinch-off → Positive-Current Region → Gain Switch → n=1 → Pump Map')
print('='*60)

In [ ]:
# Cell 2: Configuration

class Config:
    """
    Experiment Configuration for 5-Phase Workflow (v5)
    """

    # ==================== PHYSICAL CONSTANTS ====================
    e = 1.60217663e-19
    f = 8e7

    @property
    def target_current(self):
        return self.e * self.f

    # ==================== GAIN PROFILE ====================
    # Phase 1/2: 1e-7 A/V, Phase 4/5: 1e-9 A/V (after manual dial switch at Phase 3)
    GAIN_PHASE1_A_PER_V = 1e-7
    GAIN_PHASE2_A_PER_V = 1e-7
    GAIN_PHASE4_A_PER_V = 1e-9
    CURRENT_AMP_GAIN = GAIN_PHASE1_A_PER_V

    # ==================== RULE NO.1: VOLTAGE STEP ====================
    MAX_VOLTAGE_STEP_V = 0.004
    SETTLING_TIME = 0.08

    # ==================== GPIB ADDRESSES ====================
    ADDR_YOKO_ENT  = "GPIB43::1::INSTR"
    ADDR_YOKO_P    = "GPIB43::2::INSTR"
    ADDR_YOKO_EXIT = "GPIB43::8::INSTR"
    ADDR_DMM       = "GPIB43::19::INSTR"

    # ==================== PHASE 1: PINCH-OFF ====================
    PINCH_SCAN_RANGES = {
        'V_ent': (-0.3, 0.3),
        'V_p': (-0.3, 0.3),
        'V_exit': (-0.3, 0.3),
    }
    PINCH_SCAN_POINTS = 121
    PINCH_REF_V_ENT = 0.0
    PINCH_REF_V_P = 0.0
    PINCH_REF_V_EXIT = 0.0
    PINCH_OFF_THRESHOLD_A = 0.5e-9

    # ==================== PHASE 2: POSITIVE CURRENT REGION ====================
    PHASE2_RANGE_V_ENT = 0.20
    PHASE2_RANGE_V_EXIT = 0.40
    PHASE2_GRID_N = 31
    PHASE2_I_MIN_A = 0.0
    PHASE2_I_MAX_A = 4e-9

    # ==================== PHASE 4: CONSTRAINED n=1 BO ====================
    PHASE4_TARGET_N = 1.0
    PHASE4_N_INITIAL = 20
    PHASE4_N_ITER = 80
    PHASE4_EARLY_STOP_PATIENCE = 25
    PHASE4_VP_WINDOW = 0.12

    PHASE4_N_TOL = 0.03
    PHASE4_PLATEAU_TOP_K = 8
    PHASE4_DV_EXIT = 0.006
    PHASE4_FLATNESS_WEIGHT = 0.35

    # ==================== PHASE 5: PUMP MAP ====================
    MAPPING_RANGE_V_ENT = 0.2
    MAPPING_RANGE_V_EXIT = 0.40
    N_MAPPING_LHS = 30
    N_MAPPING_ADAPTIVE = 70
    MAPPING_GRID_RESOLUTION = 80
    TRANSCONDUCTANCE_CLIP_PERCENTILE = 99.0
    CURVE_OFFSETS = [-0.04, -0.02, 0.02, 0.04]

    # ==================== MODE ====================
    FORCE_SIMULATION = False

    # ==================== OUTPUT ====================
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = Path(f"./experiment_outputs_v5_{timestamp}")

    @classmethod
    def print_settings(cls):
        cfg = cls()
        print('='*60)
        print('EXPERIMENT CONFIGURATION (v5)')
        print('='*60)
        print(f'Frequency:                   {cls.f/1e9:.3f} GHz')
        print(f'Target current (n=1):        {cfg.target_current*1e12:.4f} pA (ef)')
        print(f'Max voltage step:            {cls.MAX_VOLTAGE_STEP_V*1e3:.1f} mV')

        print()
        print('--- Gain policy ---')
        print(f'Phase 1 gain:                {cls.GAIN_PHASE1_A_PER_V:.0e} A/V')
        print(f'Phase 2 gain:                {cls.GAIN_PHASE2_A_PER_V:.0e} A/V')
        print(f'Phase 4/5 gain:              {cls.GAIN_PHASE4_A_PER_V:.0e} A/V')

        print()
        print('--- Phase 1: Pinch-off ---')
        print(f'Scan points per gate:        {cls.PINCH_SCAN_POINTS}')
        print(f'Pinch-off threshold |I|:     {cls.PINCH_OFF_THRESHOLD_A*1e9:.3f} nA')

        print()
        print('--- Phase 2: Positive current region ---')
        print(f'Grid:                        {cls.PHASE2_GRID_N} x {cls.PHASE2_GRID_N}')
        print(f'Condition:                   {cls.PHASE2_I_MIN_A*1e9:.1f} < I < {cls.PHASE2_I_MAX_A*1e9:.1f} nA')
        print(f'V_ENT total range:           {cls.PHASE2_RANGE_V_ENT:.3f} V')
        print(f'V_EXIT total range:          {cls.PHASE2_RANGE_V_EXIT:.3f} V')

        print()
        print('--- Phase 4: Constrained BO ---')
        print(f'Initial points:              {cls.PHASE4_N_INITIAL}')
        print(f'Max BO iterations:           {cls.PHASE4_N_ITER}')
        print(f'Early stop patience:         {cls.PHASE4_EARLY_STOP_PATIENCE}')

        print()
        print('--- Phase 5: Pump map ---')
        print(f'LHS samples:                 {cls.N_MAPPING_LHS}')
        print(f'Adaptive samples:            {cls.N_MAPPING_ADAPTIVE}')
        print(f'Total mapping points:        {cls.N_MAPPING_LHS + cls.N_MAPPING_ADAPTIVE}')

        print()
        print(f'Force simulation:            {cls.FORCE_SIMULATION}')
        print('='*60)


Config.print_settings()

In [ ]:
# Cell 3: Instrument Controller

class InstrumentController:
    """
    Hardware interface with simulation fallback.

    Rule No.1 is enforced via set_voltages_stepwise:
    every voltage update is split so that each step <= MAX_VOLTAGE_STEP_V.
    """

    def __init__(self, config):
        self.cfg = config
        self.simulation_mode = False
        self.current_amp_gain = float(config.CURRENT_AMP_GAIN)
        self.current_voltages = {
            'V_ent': float(config.PINCH_REF_V_ENT),
            'V_p': float(config.PINCH_REF_V_P),
            'V_exit': float(config.PINCH_REF_V_EXIT),
        }

        if config.FORCE_SIMULATION:
            print('⚠️ FORCE_SIMULATION enabled')
            self.simulation_mode = True
            self._init_simulation()
            return

        if not PYVISA_AVAILABLE:
            print('⚠️ PyVISA not available')
            self.simulation_mode = True
            self._init_simulation()
            return

        try:
            self.rm = pyvisa.ResourceManager()
            print('Connecting to instruments...')

            self.yoko_ent = self.rm.open_resource(config.ADDR_YOKO_ENT)
            print(f'  ✅ G_ENT: {config.ADDR_YOKO_ENT}')

            self.yoko_p = self.rm.open_resource(config.ADDR_YOKO_P)
            print(f'  ✅ G_P: {config.ADDR_YOKO_P}')

            self.yoko_exit = self.rm.open_resource(config.ADDR_YOKO_EXIT)
            print(f'  ✅ G_EXIT: {config.ADDR_YOKO_EXIT}')

            self.dmm = self.rm.open_resource(config.ADDR_DMM)
            print(f'  ✅ DMM: {config.ADDR_DMM}')

            self._configure_instruments()
            print()
            print('✅ HARDWARE MODE')
            print(f'   Target current:      {config.target_current*1e12:.4f} pA (ef)')
            print(f'   Active amp gain:     {self.current_amp_gain:.0e} A/V')

        except Exception as e:
            print()
            print(f'⚠️ Hardware connection failed: {e}')
            print('   Using SIMULATION MODE')
            self.simulation_mode = True
            self._init_simulation()

    def _configure_instruments(self):
        for inst in [self.yoko_ent, self.yoko_p, self.yoko_exit]:
            inst.write_termination = "\n"
            inst.read_termination = "\n"
            inst.timeout = 5000
        self.dmm.write_termination = "\n"
        self.dmm.read_termination = "\n"
        self.dmm.timeout = 10000

    def _init_simulation(self):
        self.Va_base = -0.66
        self.Vb = 0.008
        self.delta2 = 0.10
        self.cross_coupling_ent = 0.30
        self.V_ent_ref = -0.62
        self.V_ent_center = -0.62
        self.ent_width = 0.20
        self.ent_sharpness = 100.0
        self.V_p_center = -0.63
        self.p_width = 0.20
        self.p_sharpness = 80.0
        self.noise_std = 1e-4
        print('   Simulation model initialized')

    def set_current_amp_gain(self, gain_a_per_v):
        self.current_amp_gain = float(gain_a_per_v)
        self.cfg.CURRENT_AMP_GAIN = float(gain_a_per_v)
        print(f'🔧 Active current amp gain updated: {self.current_amp_gain:.0e} A/V')

    def _write_voltages_direct(self, V_ent, V_p, V_exit):
        self.yoko_ent.write(f'S{V_ent:.6f}E')
        self.yoko_p.write(f'S{V_p:.6f}E')
        self.yoko_exit.write(f'S{V_exit:.6f}E')

    def set_voltages_stepwise(self, V_ent, V_p, V_exit):
        target = np.array([float(V_ent), float(V_p), float(V_exit)], dtype=float)
        start = np.array([
            self.current_voltages['V_ent'],
            self.current_voltages['V_p'],
            self.current_voltages['V_exit']
        ], dtype=float)

        max_delta = np.max(np.abs(target - start))
        n_steps = max(1, int(np.ceil(max_delta / self.cfg.MAX_VOLTAGE_STEP_V)))

        for k in range(1, n_steps + 1):
            alpha = k / n_steps
            vk = start + alpha * (target - start)

            if not self.simulation_mode:
                self._write_voltages_direct(vk[0], vk[1], vk[2])
                time.sleep(self.cfg.SETTLING_TIME)

            self.current_voltages['V_ent'] = float(vk[0])
            self.current_voltages['V_p'] = float(vk[1])
            self.current_voltages['V_exit'] = float(vk[2])

    def set_voltages(self, V_ent, V_p, V_exit):
        self.set_voltages_stepwise(V_ent, V_p, V_exit)

    def measure_current(self):
        if not self.simulation_mode:
            try:
                dmm_voltage = float(self.dmm.query('fetch?'))
                current = -dmm_voltage * self.current_amp_gain
                return current
            except Exception as e:
                print(f'⚠️ DMM error: {e}')
                return np.nan

        n = self._simulate_n(
            self.current_voltages['V_ent'],
            self.current_voltages['V_p'],
            self.current_voltages['V_exit']
        )
        return n * self.cfg.target_current

    def measure_current_at(self, V_ent, V_p, V_exit):
        self.set_voltages_stepwise(V_ent, V_p, V_exit)
        return self.measure_current()

    def measure(self, V_ent, V_p, V_exit):
        current = self.measure_current_at(V_ent, V_p, V_exit)
        return current / self.cfg.target_current

    def measure_2d(self, V_ent, V_exit, V_p_fixed):
        return self.measure(V_ent, V_p_fixed, V_exit)

    def _simulate_n(self, V_ent, V_p, V_exit):
        Va_eff = self.Va_base + self.cross_coupling_ent * (V_ent - self.V_ent_ref)
        arg1 = np.clip(-(V_exit - Va_eff) / self.Vb, -100, 100)
        arg2 = np.clip(-(V_exit - Va_eff + self.delta2) / self.Vb, -100, 100)
        f_exit = np.exp(-np.exp(arg1)) + np.exp(-np.exp(arg2))

        V_L = self.V_ent_center - self.ent_width / 2
        V_R = self.V_ent_center + self.ent_width / 2
        f_ent = self._sigmoid(self.ent_sharpness * (V_ent - V_L)) * self._sigmoid(-self.ent_sharpness * (V_ent - V_R))

        V_L_p = self.V_p_center - self.p_width / 2
        V_R_p = self.V_p_center + self.p_width / 2
        f_p = self._sigmoid(self.p_sharpness * (V_p - V_L_p)) * self._sigmoid(-self.p_sharpness * (V_p - V_R_p))

        n = f_exit * f_ent * f_p
        n += np.random.normal(0, self.noise_std)
        return n

    def _sigmoid(self, x):
        return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

    def get_ground_truth_map(self, V_ent_range, V_exit_range, V_p_fixed):
        if not self.simulation_mode:
            return None, None, None

        V_ENT, V_EXIT = np.meshgrid(V_ent_range, V_exit_range)
        n_map = np.zeros_like(V_ENT)

        noise_backup = self.noise_std
        self.noise_std = 0
        for i in range(len(V_exit_range)):
            for j in range(len(V_ent_range)):
                n_map[i, j] = self._simulate_n(V_ENT[i, j], V_p_fixed, V_EXIT[i, j])
        self.noise_std = noise_backup

        return V_ENT, V_EXIT, n_map

    def close(self):
        if not self.simulation_mode:
            try:
                self.yoko_ent.close()
                self.yoko_p.close()
                self.yoko_exit.close()
                self.dmm.close()
                self.rm.close()
                print('Instruments disconnected')
            except Exception:
                pass

    def get_mode_string(self):
        return 'SIMULATION' if self.simulation_mode else 'HARDWARE'


print('InstrumentController defined')

In [ ]:
# Cell 4: Bayesian Optimizer (for Phase 4)

class BayesianOptimizer:
    """BO with EI acquisition for Phase 1 optimization."""
    
    def __init__(self, bounds):
        self.bounds = bounds
        self.dim = len(bounds)
        
        self.kernel = ConstantKernel(1.0) * Matern(length_scale=0.1, nu=2.5) + \
                      WhiteKernel(noise_level=1e-5)
        
        self.gp_cost = GaussianProcessRegressor(
            kernel=self.kernel, n_restarts_optimizer=10, normalize_y=True)
        self.gp_n = GaussianProcessRegressor(
            kernel=self.kernel, n_restarts_optimizer=10, normalize_y=True)
        
        self.X_train = None
        self.y_train = None
        self.n_train = None
        self.is_fitted = False
    
    def fit(self, X, y_cost, n_values):
        self.X_train = np.array(X)
        self.y_train = np.array(y_cost)
        self.n_train = np.array(n_values)
        self.gp_cost.fit(self.X_train, self.y_train)
        self.gp_n.fit(self.X_train, self.n_train)
        self.is_fitted = True
    
    def _expected_improvement(self, X):
        X = np.atleast_2d(X)
        mu, sigma = self.gp_cost.predict(X, return_std=True)
        sigma = np.maximum(sigma, 1e-9)
        y_best = np.min(self.y_train)
        Z = (y_best - mu) / sigma
        ei = (y_best - mu) * norm.cdf(Z) + sigma * norm.pdf(Z)
        return -ei
    
    def suggest_next_point(self, n_candidates=5000):
        if not self.is_fitted:
            return np.random.uniform(self.bounds[:, 0], self.bounds[:, 1])
        
        X_candidates = np.random.uniform(
            self.bounds[:, 0], self.bounds[:, 1], size=(n_candidates, self.dim))
        ei_values = self._expected_improvement(X_candidates)
        best_idx = np.argmin(ei_values)
        x0 = X_candidates[best_idx]
        
        result = minimize(
            lambda x: self._expected_improvement(x.reshape(1, -1))[0],
            x0=x0, bounds=self.bounds, method='L-BFGS-B')
        return result.x
    
    def predict_cost(self, X):
        return self.gp_cost.predict(np.atleast_2d(X), return_std=True)
    
    def predict_n(self, X):
        return self.gp_n.predict(np.atleast_2d(X), return_std=True)


class EarlyStopping:
    def __init__(self, patience=25):
        self.patience = patience
        self.best_cost = np.inf
        self.counter = 0
    
    def update(self, cost):
        if cost < self.best_cost - 1e-6:
            self.best_cost = cost
            self.counter = 0
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience


print('BayesianOptimizer and EarlyStopping defined')

In [ ]:
# Cell 5: Efficient Mapper (for Phase 5)

class EfficientMapper:
    """GP-based efficient mapping for Phase 2."""
    
    def __init__(self, bounds_2d):
        self.bounds = bounds_2d  # [[V_ENT_min, V_ENT_max], [V_EXIT_min, V_EXIT_max]]
        
        self.kernel = ConstantKernel(1.0) * Matern(length_scale=0.05, nu=2.5) + \
                      WhiteKernel(noise_level=1e-6)
        self.gp = GaussianProcessRegressor(
            kernel=self.kernel, n_restarts_optimizer=10, normalize_y=True)
        
        self.X_measured = []
        self.y_measured = []
        self.is_fitted = False
    
    def generate_lhs_samples(self, n_samples):
        """Generate Latin Hypercube samples."""
        sampler = qmc.LatinHypercube(d=2, seed=42)
        samples = sampler.random(n=n_samples)
        lower = self.bounds[:, 0]
        upper = self.bounds[:, 1]
        return qmc.scale(samples, lower, upper)
    
    def fit(self):
        if len(self.X_measured) > 0:
            X = np.array(self.X_measured)
            y = np.array(self.y_measured)
            self.gp.fit(X, y)
            self.is_fitted = True
    
    def suggest_next_point(self, n_candidates=1000):
        """Suggest point with maximum uncertainty."""
        if not self.is_fitted:
            return None
        
        candidates = np.random.uniform(
            self.bounds[:, 0], self.bounds[:, 1], size=(n_candidates, 2))
        _, sigma = self.gp.predict(candidates, return_std=True)
        return candidates[np.argmax(sigma)]
    
    def add_measurement(self, x, y):
        self.X_measured.append(x)
        self.y_measured.append(y)
    
    def predict_map(self, V_ent_range, V_exit_range):
        """Predict full 2D map."""
        V_ENT, V_EXIT = np.meshgrid(V_ent_range, V_exit_range)
        X_grid = np.column_stack([V_ENT.ravel(), V_EXIT.ravel()])
        n_mean, n_std = self.gp.predict(X_grid, return_std=True)
        return V_ENT, V_EXIT, n_mean.reshape(V_ENT.shape), n_std.reshape(V_ENT.shape)
    
    def predict_curve(self, V_exit_range, V_ent_fixed):
        """Predict pumping curve at fixed V_ENT."""
        X = np.column_stack([np.full_like(V_exit_range, V_ent_fixed), V_exit_range])
        return self.gp.predict(X, return_std=True)


print('EfficientMapper defined')

In [ ]:
# Cell 6: Phase 1 (Pinch-off) + Phase 4 (Constrained BO for n=1)


def _find_threshold_crossing(voltage_axis, abs_current_axis, threshold_a):
    """Find first crossing where |I| <= threshold. Fallback: argmin(|I|)."""
    below = abs_current_axis <= threshold_a
    if np.any(below):
        idx1 = int(np.argmax(below))
        if idx1 == 0:
            return float(voltage_axis[idx1]), idx1, idx1, True
        idx0 = idx1 - 1
        x0, x1 = voltage_axis[idx0], voltage_axis[idx1]
        y0, y1 = abs_current_axis[idx0], abs_current_axis[idx1]
        if np.isclose(y1, y0):
            x_cross = x1
        else:
            x_cross = x0 + (threshold_a - y0) * (x1 - x0) / (y1 - y0)
        return float(x_cross), idx0, idx1, True

    idx = int(np.argmin(abs_current_axis))
    return float(voltage_axis[idx]), idx, idx, False


def run_phase1_pinchoff(instr, config):
    """
    Phase 1: find pinch-off voltages of V_ent, V_p, V_exit at gain=1e-7 A/V.
    """
    print()
    print('='*70)
    print('PHASE 1: PINCH-OFF SEARCH')
    print('Gain set to 1e-7 A/V, sweep each gate independently')
    print('='*70)

    instr.set_current_amp_gain(config.GAIN_PHASE1_A_PER_V)

    refs = {
        'V_ent': float(config.PINCH_REF_V_ENT),
        'V_p': float(config.PINCH_REF_V_P),
        'V_exit': float(config.PINCH_REF_V_EXIT),
    }

    gate_keys = ['V_ent', 'V_p', 'V_exit']
    traces = {}
    pinch_off = {}
    total_measurements = 0

    for gate in gate_keys:
        vmin, vmax = config.PINCH_SCAN_RANGES[gate]
        sweep = np.linspace(vmin, vmax, config.PINCH_SCAN_POINTS)
        currents = []

        print()
        print(f'[Phase1] Sweeping {gate}: {vmin:.3f} -> {vmax:.3f} V ({config.PINCH_SCAN_POINTS} points)')

        for i, vg in enumerate(sweep):
            V_ent = refs['V_ent']
            V_p = refs['V_p']
            V_exit = refs['V_exit']

            if gate == 'V_ent':
                V_ent = vg
            elif gate == 'V_p':
                V_p = vg
            else:
                V_exit = vg

            current = instr.measure_current_at(V_ent, V_p, V_exit)
            currents.append(current)
            total_measurements += 1

            if (i + 1) % 30 == 0 or i == len(sweep) - 1:
                print(f'  {gate} {i+1:3d}/{len(sweep)}: V={vg:+.4f} V, I={current*1e9:+.4f} nA')

        currents = np.asarray(currents, dtype=float)
        abs_currents = np.abs(currents)

        pinch_v, idx0, idx1, crossed = _find_threshold_crossing(
            sweep, abs_currents, config.PINCH_OFF_THRESHOLD_A
        )

        pinch_off[gate] = float(pinch_v)
        traces[gate] = {
            'voltage': sweep,
            'current_a': currents,
            'abs_current_a': abs_currents,
            'cross_idx0': int(idx0),
            'cross_idx1': int(idx1),
            'crossed_threshold': bool(crossed),
            'pinch_off_v': float(pinch_v),
        }

        flag = 'threshold-cross' if crossed else 'fallback-min|I|'
        print(f'  -> Pinch-off {gate}: {pinch_v:+.6f} V ({flag})')

    results = {
        'phase': 'phase1_pinchoff',
        'gain_a_per_v': float(instr.current_amp_gain),
        'pinch_threshold_a': float(config.PINCH_OFF_THRESHOLD_A),
        'pinch_off_voltages': pinch_off,
        'traces': traces,
        'total_measurements': int(total_measurements),
        'refs': refs,
    }
    return results


def display_phase1_results(results, config):
    print()
    print('='*70)
    print('PHASE 1 COMPLETE - PINCH-OFF RESULTS')
    print('='*70)
    print(f'Gain used:        {results["gain_a_per_v"]:.0e} A/V')
    print(f'|I| threshold:    {results["pinch_threshold_a"]*1e9:.3f} nA')
    print(f'Measurements:     {results["total_measurements"]}')

    po = results['pinch_off_voltages']
    print()
    print('Pinch-off voltages:')
    print(f'  V_ENT pinch-off:  {po["V_ent"]:+.6f} V')
    print(f'  V_P pinch-off:    {po["V_p"]:+.6f} V')
    print(f'  V_EXIT pinch-off: {po["V_exit"]:+.6f} V')

    half_ent = config.PHASE2_RANGE_V_ENT / 2
    half_exit = config.PHASE2_RANGE_V_EXIT / 2
    print()
    print('Proposed Phase 2 scan window around pinch-off center:')
    print(f'  V_ENT:  [{po["V_ent"]-half_ent:+.3f}, {po["V_ent"]+half_ent:+.3f}] V')
    print(f'  V_EXIT: [{po["V_exit"]-half_exit:+.3f}, {po["V_exit"]+half_exit:+.3f}] V')
    print(f'  Fixed V_P = {po["V_p"]:+.4f} V')



def run_phase4_optimization(instr, phase1_results, phase2_selected, config):
    """
    Phase 4: constrained BO to find n=1 region.
    Bounds are constrained by selected Phase 2 valid region.
    """
    print()
    print('='*70)
    print('PHASE 4: CONSTRAINED BO FOR n=1')
    print('='*70)

    valid_bounds = np.asarray(phase2_selected['valid_bounds'], dtype=float)
    V_ent_bounds = valid_bounds[0]
    V_exit_bounds = valid_bounds[1]

    V_p_center = phase1_results['pinch_off_voltages']['V_p']
    half_vp = config.PHASE4_VP_WINDOW / 2
    V_p_bounds = np.array([V_p_center - half_vp, V_p_center + half_vp], dtype=float)

    bounds_optimization = np.array([
        [V_ent_bounds[0], V_ent_bounds[1]],
        [V_p_bounds[0], V_p_bounds[1]],
        [V_exit_bounds[0], V_exit_bounds[1]],
    ], dtype=float)

    print('Constrained BO bounds:')
    print(f'  V_ENT:  [{bounds_optimization[0,0]:+.4f}, {bounds_optimization[0,1]:+.4f}] V')
    print(f'  V_P:    [{bounds_optimization[1,0]:+.4f}, {bounds_optimization[1,1]:+.4f}] V')
    print(f'  V_EXIT: [{bounds_optimization[2,0]:+.4f}, {bounds_optimization[2,1]:+.4f}] V')

    bo = BayesianOptimizer(bounds_optimization)
    early_stop = EarlyStopping(patience=config.PHASE4_EARLY_STOP_PATIENCE)

    X_hist, y_hist, n_hist = [], [], []

    print()
    print(f'{"Iter":>5} {"Phase":>6} {"V_ENT":>9} {"V_P":>9} {"V_EXIT":>9} {"n":>10} {"Cost":>8} {"Best":>8}')
    print('-'*75)

    total_iterations = config.PHASE4_N_INITIAL + config.PHASE4_N_ITER
    stopped_early = False
    early_stop_iteration = None

    for i in range(total_iterations):
        if i < config.PHASE4_N_INITIAL:
            phase = 'Init'
            x_next = np.random.uniform(bounds_optimization[:, 0], bounds_optimization[:, 1])
        else:
            phase = 'BO'
            bo.fit(X_hist, y_hist, n_hist)
            x_next = bo.suggest_next_point()

        V_ent, V_p, V_exit = x_next
        n = instr.measure(V_ent, V_p, V_exit)
        cost = np.log10(np.abs(n - config.PHASE4_TARGET_N) + 1e-12)

        X_hist.append(x_next)
        y_hist.append(cost)
        n_hist.append(n)

        best_cost = np.min(y_hist)
        print(f'{i:>5} {phase:>6} {V_ent:>9.4f} {V_p:>9.4f} {V_exit:>9.4f} {n:>10.5f} {cost:>8.2f} {best_cost:>8.2f}')

        if phase == 'BO':
            if early_stop.update(cost):
                early_stop_iteration = i
                stopped_early = True
                print()
                print(f'⚠️ Early stopping at iteration {i}')
                break

    bo.fit(X_hist, y_hist, n_hist)

    X_hist_arr = np.array(X_hist)
    y_hist_arr = np.array(y_hist)
    n_hist_arr = np.array(n_hist)

    best_idx_by_cost = int(np.argmin(y_hist_arr))
    best_idx = best_idx_by_cost

    plateau_scan = []
    phase4_refinement_measurements = 0
    phase4_refinement_used = False

    top_k = min(config.PHASE4_PLATEAU_TOP_K, len(X_hist_arr))
    if top_k > 0:
        n_error = np.abs(n_hist_arr - config.PHASE4_TARGET_N)
        candidate_idx = np.argsort(n_error)[:top_k]

        near_plateau_idx = candidate_idx[n_error[candidate_idx] <= config.PHASE4_N_TOL]
        if len(near_plateau_idx) > 0:
            candidate_idx = near_plateau_idx

        v_exit_min, v_exit_max = bounds_optimization[2]

        for idx in candidate_idx:
            V_ent, V_p, V_exit = X_hist_arr[idx]
            n_center = n_hist_arr[idx]

            v_minus = np.clip(V_exit - config.PHASE4_DV_EXIT, v_exit_min, v_exit_max)
            v_plus = np.clip(V_exit + config.PHASE4_DV_EXIT, v_exit_min, v_exit_max)
            if np.isclose(v_plus, v_minus):
                continue

            n_minus = instr.measure(V_ent, V_p, v_minus)
            n_plus = instr.measure(V_ent, V_p, v_plus)
            phase4_refinement_measurements += 2

            dn_dV_exit = (n_plus - n_minus) / (v_plus - v_minus)
            flatness = np.abs(n_plus - n_minus)
            n_error_center = np.abs(n_center - config.PHASE4_TARGET_N)
            plateau_score = n_error_center + config.PHASE4_FLATNESS_WEIGHT * flatness

            plateau_scan.append({
                'idx': int(idx),
                'V_ent': float(V_ent),
                'V_p': float(V_p),
                'V_exit': float(V_exit),
                'n_center': float(n_center),
                'n_minus': float(n_minus),
                'n_plus': float(n_plus),
                'dn_dV_exit': float(dn_dV_exit),
                'flatness': float(flatness),
                'n_error': float(n_error_center),
                'plateau_score': float(plateau_score),
            })

    if plateau_scan:
        best_plateau_row = min(plateau_scan, key=lambda row: row['plateau_score'])
        best_idx = int(best_plateau_row['idx'])
        phase4_refinement_used = True

    best_x = X_hist_arr[best_idx]
    best_n = n_hist_arr[best_idx]
    best_cost = y_hist_arr[best_idx]

    best_dn_dV_exit = np.nan
    best_plateau_score = np.nan
    for row in plateau_scan:
        if row['idx'] == best_idx:
            best_dn_dV_exit = row['dn_dV_exit']
            best_plateau_score = row['plateau_score']
            break

    results = {
        'phase': 'phase4_n1_optimization',
        'bounds_optimization': bounds_optimization,
        'X_hist': X_hist_arr,
        'y_hist': y_hist_arr,
        'n_hist': n_hist_arr,
        'best_V_ent': float(best_x[0]),
        'best_V_p': float(best_x[1]),
        'best_V_exit': float(best_x[2]),
        'best_n': float(best_n),
        'best_cost': float(best_cost),
        'best_dn_dV_exit': float(best_dn_dV_exit),
        'best_plateau_score': float(best_plateau_score),
        'best_idx': int(best_idx),
        'best_idx_by_cost': int(best_idx_by_cost),
        'phase4_target_n': float(config.PHASE4_TARGET_N),
        'phase1_target_n': float(config.PHASE4_TARGET_N),
        'phase4_refinement_used': phase4_refinement_used,
        'phase4_refinement_candidates': len(plateau_scan),
        'phase4_refinement_measurements': int(phase4_refinement_measurements),
        'total_measurements_including_refinement': int(len(X_hist_arr) + phase4_refinement_measurements),
        'plateau_scan': plateau_scan,
        'bo': bo,
        'stopped_early': bool(stopped_early),
        'early_stop_iteration': early_stop_iteration,
        'total_iterations': len(X_hist_arr),
        'selected_phase2_attempt': int(phase2_selected['attempt_idx']),
    }
    return results


def display_phase4_results(results, config):
    print()
    print('='*70)
    print('PHASE 4 COMPLETE - n=1 OPTIMIZATION RESULTS')
    print('='*70)

    print(f'Total measurements: {results["total_measurements_including_refinement"]}')
    print(f'Best V_ENT:         {results["best_V_ent"]:+.6f} V')
    print(f'Best V_P:           {results["best_V_p"]:+.6f} V')
    print(f'Best V_EXIT:        {results["best_V_exit"]:+.6f} V')
    print(f'Best n:             {results["best_n"]:.6f}')
    print(f'Best |n-1|:         {np.abs(results["best_n"] - results["phase4_target_n"]):.3e}')

    if results['phase4_refinement_used']:
        print(f'Refinement used:    Yes ({results["phase4_refinement_candidates"]} candidates)')
    else:
        print('Refinement used:    No')

In [ ]:
# Cell 7: Phase 2 (Positive Current Region) + Phase 5 (Pump Map)


def _largest_connected_component(mask):
    if not np.any(mask):
        return np.zeros_like(mask, dtype=bool)
    labeled, n_comp = label(mask.astype(int))
    if n_comp <= 0:
        return np.zeros_like(mask, dtype=bool)
    sizes = [np.sum(labeled == i) for i in range(1, n_comp + 1)]
    best_label = 1 + int(np.argmax(sizes))
    return labeled == best_label


def run_phase2_positive_current_mapping(instr, phase1_results, config, attempt_idx=1):
    """
    Phase 2: 2D scan to find region satisfying 0 < I < 4 nA.
    Gain remains at 1e-7 A/V.
    """
    print()
    print('='*70)
    print(f'PHASE 2: POSITIVE CURRENT REGION SEARCH (Attempt {attempt_idx})')
    print('='*70)

    instr.set_current_amp_gain(config.GAIN_PHASE2_A_PER_V)

    po = phase1_results['pinch_off_voltages']
    V_ent_center = float(po['V_ent'])
    V_exit_center = float(po['V_exit'])
    V_p_fixed = float(po['V_p'])

    half_ent = config.PHASE2_RANGE_V_ENT / 2
    half_exit = config.PHASE2_RANGE_V_EXIT / 2

    bounds_2d_default = np.array([
        [V_ent_center - half_ent, V_ent_center + half_ent],
        [V_exit_center - half_exit, V_exit_center + half_exit],
    ], dtype=float)

    V_ent_range = np.linspace(bounds_2d_default[0, 0], bounds_2d_default[0, 1], config.PHASE2_GRID_N)
    V_exit_range = np.linspace(bounds_2d_default[1, 0], bounds_2d_default[1, 1], config.PHASE2_GRID_N)

    current_map_a = np.zeros((len(V_exit_range), len(V_ent_range)), dtype=float)

    total_pts = current_map_a.size
    cnt = 0
    for i, V_exit in enumerate(V_exit_range):
        for j, V_ent in enumerate(V_ent_range):
            current_map_a[i, j] = instr.measure_current_at(V_ent, V_p_fixed, V_exit)
            cnt += 1
        if (i + 1) % 5 == 0 or (i + 1) == len(V_exit_range):
            print(f'  row {i+1:2d}/{len(V_exit_range)} complete ({cnt}/{total_pts} points)')

    valid_raw = (current_map_a > config.PHASE2_I_MIN_A) & (current_map_a < config.PHASE2_I_MAX_A)
    valid_mask = _largest_connected_component(valid_raw)

    if np.any(valid_mask):
        idx = np.argwhere(valid_mask)
        i_exit = idx[:, 0]
        i_ent = idx[:, 1]

        valid_bounds = np.array([
            [np.min(V_ent_range[i_ent]), np.max(V_ent_range[i_ent])],
            [np.min(V_exit_range[i_exit]), np.max(V_exit_range[i_exit])],
        ], dtype=float)
        used_fallback_bounds = False
    else:
        valid_bounds = bounds_2d_default.copy()
        used_fallback_bounds = True

    results = {
        'phase': 'phase2_positive_region',
        'attempt_idx': int(attempt_idx),
        'gain_a_per_v': float(instr.current_amp_gain),
        'V_p_fixed': float(V_p_fixed),
        'V_ent_center': float(V_ent_center),
        'V_exit_center': float(V_exit_center),
        'bounds_2d_default': bounds_2d_default,
        'valid_bounds': valid_bounds,
        'V_ent_range': V_ent_range,
        'V_exit_range': V_exit_range,
        'current_map_a': current_map_a,
        'current_map_na': current_map_a * 1e9,
        'valid_mask_raw': valid_raw,
        'valid_mask': valid_mask,
        'valid_points_raw': int(np.sum(valid_raw)),
        'valid_points_connected': int(np.sum(valid_mask)),
        'used_fallback_bounds': bool(used_fallback_bounds),
        'i_condition_min_a': float(config.PHASE2_I_MIN_A),
        'i_condition_max_a': float(config.PHASE2_I_MAX_A),
    }

    print()
    print('Phase 2 result summary:')
    print(f'  Gain:                    {results["gain_a_per_v"]:.0e} A/V')
    print(f'  Valid points (raw):      {results["valid_points_raw"]}')
    print(f'  Valid points (largest):  {results["valid_points_connected"]}')
    print(f'  Using fallback bounds:   {results["used_fallback_bounds"]}')
    print(f'  Valid V_ENT bounds:      [{results["valid_bounds"][0,0]:+.4f}, {results["valid_bounds"][0,1]:+.4f}] V')
    print(f'  Valid V_EXIT bounds:     [{results["valid_bounds"][1,0]:+.4f}, {results["valid_bounds"][1,1]:+.4f}] V')

    return results


def run_phase5_mapping(instr, phase4_results, config):
    """
    Phase 5: existing 6-panel pump map workflow (LHS + adaptive GP mapping).
    """
    print()
    print('='*70)
    print('PHASE 5: PUMP MAP MAPPING')
    print('Mapping pump characteristics around Phase 4 optimum')
    print('='*70)

    half_V_ent = config.MAPPING_RANGE_V_ENT / 2
    half_V_exit = config.MAPPING_RANGE_V_EXIT / 2

    V_ent_center = phase4_results['best_V_ent']
    V_p_fixed = phase4_results['best_V_p']
    V_exit_center = phase4_results['best_V_exit']

    bounds_2d = np.array([
        [V_ent_center - half_V_ent, V_ent_center + half_V_ent],
        [V_exit_center - half_V_exit, V_exit_center + half_V_exit]
    ])

    print()
    print('Mapping parameters:')
    print(f'  Center V_ENT:  {V_ent_center:.4f} V')
    print(f'  Center V_EXIT: {V_exit_center:.4f} V')
    print(f'  Fixed V_P:     {V_p_fixed:.4f} V')
    print(f'  V_ENT range:   [{bounds_2d[0,0]:.3f}, {bounds_2d[0,1]:.3f}] V  (total {config.MAPPING_RANGE_V_ENT:.3f} V)')
    print(f'  V_EXIT range:  [{bounds_2d[1,0]:.3f}, {bounds_2d[1,1]:.3f}] V  (total {config.MAPPING_RANGE_V_EXIT:.3f} V)')

    mapper = EfficientMapper(bounds_2d)

    print()
    print(f'Phase 5a: Latin Hypercube Sampling ({config.N_MAPPING_LHS} points)')
    print('-'*50)
    lhs_points = mapper.generate_lhs_samples(config.N_MAPPING_LHS)

    for i, (V_ent, V_exit) in enumerate(lhs_points):
        n = instr.measure_2d(V_ent, V_exit, V_p_fixed)
        mapper.add_measurement([V_ent, V_exit], n)

        if (i + 1) % 10 == 0:
            print(f'  LHS {i+1:3d}/{config.N_MAPPING_LHS}: V_ENT={V_ent:.3f}V, V_EXIT={V_exit:.3f}V, n={n:.4f}')

    mapper.fit()
    print(f'  GP fitted with {len(mapper.X_measured)} points')

    print()
    print(f'Phase 5b: Adaptive Uncertainty Sampling ({config.N_MAPPING_ADAPTIVE} points)')
    print('-'*50)

    for i in range(config.N_MAPPING_ADAPTIVE):
        next_point = mapper.suggest_next_point()
        V_ent, V_exit = next_point

        n = instr.measure_2d(V_ent, V_exit, V_p_fixed)
        mapper.add_measurement([V_ent, V_exit], n)
        mapper.fit()

        if (i + 1) % 10 == 0:
            V_ent_test = np.linspace(bounds_2d[0, 0], bounds_2d[0, 1], 20)
            V_exit_test = np.linspace(bounds_2d[1, 0], bounds_2d[1, 1], 20)
            _, _, _, sigma_map = mapper.predict_map(V_ent_test, V_exit_test)
            mean_sigma = np.mean(sigma_map)

            print(f'  Adaptive {i+1:3d}/{config.N_MAPPING_ADAPTIVE}: V_ENT={V_ent:.3f}V, V_EXIT={V_exit:.3f}V, n={n:.4f}, mean_σ={mean_sigma:.4f}')

    print()
    print(f'Mapping complete: {len(mapper.X_measured)} total measurements')

    results = {
        'phase': 'phase5_pump_map',
        'mapper': mapper,
        'bounds_2d': bounds_2d,
        'V_p_fixed': V_p_fixed,
        'V_ent_center': V_ent_center,
        'V_exit_center': V_exit_center,
        'X_measured': np.array(mapper.X_measured),
        'n_measured': np.array(mapper.y_measured)
    }

    return results


print('Phase 2 and Phase 5 functions defined')

In [ ]:
# Cell 8: Visualization Functions


def _symmetric_limit(values, percentile=99.0, fallback=1e-6):
    arr = np.asarray(values)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return fallback

    lim = np.percentile(np.abs(arr), percentile)
    if not np.isfinite(lim) or lim <= 0:
        lim = np.max(np.abs(arr))
    return max(float(lim), fallback)


def plot_phase1_pinchoff(phase1_results, config):
    """Plot gate-wise I-V traces and pinch-off points."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    gates = ['V_ent', 'V_p', 'V_exit']
    titles = ['V_ENT sweep', 'V_P sweep', 'V_EXIT sweep']

    thr_na = phase1_results['pinch_threshold_a'] * 1e9

    for ax, gate, title in zip(axes, gates, titles):
        tr = phase1_results['traces'][gate]
        v = tr['voltage']
        i_na = tr['current_a'] * 1e9
        po = tr['pinch_off_v']

        ax.plot(v, i_na, 'b-', lw=1.8)
        ax.axhline(+thr_na, color='r', ls='--', lw=1)
        ax.axhline(-thr_na, color='r', ls='--', lw=1, label='|I| threshold' if gate == 'V_ent' else None)
        ax.axvline(po, color='k', ls=':', lw=1.5, label='pinch-off' if gate == 'V_ent' else None)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel('Voltage (V)')
        ax.set_ylabel('Current (nA)')
        ax.grid(True, alpha=0.3)

    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc='upper right')

    plt.suptitle('Phase 1: Pinch-off Search', fontsize=13, fontweight='bold')
    plt.tight_layout()

    config.output_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filepath = config.output_dir / f'phase1_pinchoff_{timestamp}.png'
    fig.savefig(filepath, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {filepath.resolve()}')
    return fig


def plot_phase2_positive_region(phase2_results, config):
    """Review plot for 0 < I < 4 nA region."""
    fig, ax = plt.subplots(1, 1, figsize=(7.5, 6.2))

    V_ent = phase2_results['V_ent_range']
    V_exit = phase2_results['V_exit_range']
    V_ENT, V_EXIT = np.meshgrid(V_ent, V_exit)
    i_na = phase2_results['current_map_na']

    cm = ax.pcolormesh(V_ENT, V_EXIT, i_na, cmap='RdYlBu_r', shading='auto')
    cbar = plt.colorbar(cm, ax=ax)
    cbar.set_label('Current (nA)')

    mask = phase2_results['valid_mask']
    if np.any(mask):
        ax.contour(V_ENT, V_EXIT, mask.astype(float), levels=[0.5], colors='lime', linewidths=1.8)

    vb = phase2_results['valid_bounds']
    ax.plot([vb[0,0], vb[0,1], vb[0,1], vb[0,0], vb[0,0]],
            [vb[1,0], vb[1,0], vb[1,1], vb[1,1], vb[1,0]],
            'w--', lw=1.5, label='Selected bounds')

    ax.set_xlabel('V_ENT (V)')
    ax.set_ylabel('V_EXIT (V)')
    ax.set_title(f'Phase 2 Attempt {phase2_results["attempt_idx"]}: 0 < I < 4 nA region')
    ax.legend(loc='upper right')
    ax.grid(alpha=0.2)

    plt.tight_layout()

    config.output_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filepath = config.output_dir / f'phase2_positive_region_attempt{phase2_results["attempt_idx"]}_{timestamp}.png'
    fig.savefig(filepath, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {filepath.resolve()}')
    return fig


def plot_phase4_summary(phase4_results, instr, config):
    """Quick summary plot for Phase 4 optimization."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    y_hist = phase4_results['y_hist']
    n_hist = phase4_results['n_hist']
    target_n = phase4_results.get('phase4_target_n', 1.0)

    ax = axes[0]
    ax.plot(y_hist, 'o-', color='steelblue', ms=4, alpha=0.6, label='Cost')
    ax.plot(np.minimum.accumulate(y_hist), 'r-', lw=2, label='Best so far')
    ax.axvline(config.PHASE4_N_INITIAL, color='gray', ls='--', alpha=0.5, label='Init→BO')
    ax.set_xlabel('Iteration', fontsize=12)
    ax.set_ylabel(r'Cost $\log_{10}|n-1|$', fontsize=12)
    ax.set_title('Phase 4: Optimization History', fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.hist(n_hist, bins=30, color='steelblue', edgecolor='k', alpha=0.7)
    ax.axvline(target_n, color='red', ls='--', lw=2, label='n=1 (target)')
    ax.axvline(phase4_results['best_n'], color='green', ls='-', lw=2,
               label=f'Best n={phase4_results["best_n"]:.4f}')
    ax.set_xlabel(r'$n$', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title('Phase 4: Distribution of n', fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.suptitle(f'Phase 4 Summary [{instr.get_mode_string()}]', fontsize=14, fontweight='bold')
    plt.tight_layout()

    config.output_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filepath = config.output_dir / f'phase4_summary_{timestamp}.png'
    fig.savefig(filepath, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {filepath.resolve()}')
    return fig


def plot_phase2_maps(phase2_results, instr, config):
    """6-panel pump map visualization (reused for Phase 5)."""
    fig = plt.figure(figsize=(18, 11))
    gs = gridspec.GridSpec(2, 3, hspace=0.3, wspace=0.3)

    mapper = phase2_results['mapper']
    bounds_2d = phase2_results['bounds_2d']
    V_p_fixed = phase2_results['V_p_fixed']
    X_meas = phase2_results['X_measured']
    n_meas = phase2_results['n_measured']

    res = config.MAPPING_GRID_RESOLUTION
    V_ent_range = np.linspace(bounds_2d[0, 0], bounds_2d[0, 1], res)
    V_exit_range = np.linspace(bounds_2d[1, 0], bounds_2d[1, 1], res)

    V_ENT, V_EXIT, n_gp_mean, n_gp_std = mapper.predict_map(V_ent_range, V_exit_range)

    dn_dV_exit_gp = np.gradient(n_gp_mean, V_exit_range, axis=0)
    dI_dV_exit_gp = config.target_current * dn_dV_exit_gp
    dI_dV_exit_gp_plot = dI_dV_exit_gp * 1e12

    tc_lim = _symmetric_limit(
        dI_dV_exit_gp_plot,
        percentile=config.TRANSCONDUCTANCE_CLIP_PERCENTILE,
        fallback=1e-3
    )

    _, _, n_truth = instr.get_ground_truth_map(V_ent_range, V_exit_range, V_p_fixed)

    mode_str = instr.get_mode_string()

    ax1 = fig.add_subplot(gs[0, 0])
    cm1 = ax1.pcolormesh(
        V_ENT,
        V_EXIT,
        dI_dV_exit_gp_plot,
        cmap='RdBu_r',
        shading='auto',
        vmin=-tc_lim,
        vmax=tc_lim
    )
    fig.colorbar(cm1, ax=ax1, label=r'$dI/dV_{EXIT}$ (pA/V)')
    ax1.scatter(
        phase2_results['V_ent_center'],
        phase2_results['V_exit_center'],
        marker='*', s=150, c='gold', edgecolor='k', linewidth=1.0,
        label='Center'
    )
    ax1.set_xlabel('$V_{ENT}$ (V)')
    ax1.set_ylabel('$V_{EXIT}$ (V)')
    ax1.set_title(f'(a) Pump Map: $dI/dV_{{EXIT}}$ (GP)\n$V_P$={V_p_fixed:.3f}V', fontsize=10)
    ax1.legend(loc='upper right', fontsize=8)

    ax2 = fig.add_subplot(gs[0, 1])
    cm2 = ax2.pcolormesh(V_ENT, V_EXIT, n_gp_mean, cmap='RdYlBu_r', shading='auto', vmin=0, vmax=2)
    fig.colorbar(cm2, ax=ax2, label='n (GP mean)')
    ax2.scatter(X_meas[:, 0], X_meas[:, 1], s=8, c='k', alpha=0.35)
    ax2.axhline(phase2_results['V_exit_center'], color='k', ls='--', lw=1, alpha=0.5)
    ax2.axvline(phase2_results['V_ent_center'], color='k', ls='--', lw=1, alpha=0.5)
    ax2.set_xlabel('$V_{ENT}$ (V)')
    ax2.set_ylabel('$V_{EXIT}$ (V)')
    ax2.set_title('(b) Pump Map: n (GP mean)', fontsize=10)

    ax3 = fig.add_subplot(gs[0, 2])
    if n_truth is not None:
        dn_dV_exit_truth = np.gradient(n_truth, V_exit_range, axis=0)
        dI_dV_exit_truth_plot = config.target_current * dn_dV_exit_truth * 1e12
        cm3 = ax3.pcolormesh(
            V_ENT,
            V_EXIT,
            dI_dV_exit_truth_plot,
            cmap='RdBu_r',
            shading='auto',
            vmin=-tc_lim,
            vmax=tc_lim
        )
        fig.colorbar(cm3, ax=ax3, label=r'$dI/dV_{EXIT}$ truth (pA/V)')
        ax3.set_title('(c) Ground Truth (Simulation)', fontsize=10)
    else:
        cm3 = ax3.pcolormesh(V_ENT, V_EXIT, n_gp_std, cmap='hot_r', shading='auto')
        fig.colorbar(cm3, ax=ax3, label='GP std')
        ax3.set_title('(c) GP Uncertainty (Hardware)', fontsize=10)
    ax3.set_xlabel('$V_{ENT}$ (V)')
    ax3.set_ylabel('$V_{EXIT}$ (V)')

    ax4 = fig.add_subplot(gs[1, 0])
    if n_truth is not None:
        delta_gp = np.abs(n_gp_mean - n_truth)
    else:
        delta_gp = np.clip(n_gp_std, 1e-6, None)
    cm4 = ax4.pcolormesh(V_ENT, V_EXIT, delta_gp, cmap='hot_r', shading='auto',
                         norm=LogNorm(vmin=1e-6, vmax=1))
    fig.colorbar(cm4, ax=ax4, label='|Error| / Proxy')
    ax4.scatter(phase2_results['V_ent_center'], phase2_results['V_exit_center'],
                marker='*', s=120, c='cyan', edgecolor='k')
    ax4.set_xlabel('$V_{ENT}$ (V)')
    ax4.set_ylabel('$V_{EXIT}$ (V)')
    ax4.set_title('(d) Error/Uncertainty map (log)', fontsize=10)

    ax5 = fig.add_subplot(gs[1, 1])
    V_exit_curve = np.linspace(bounds_2d[1, 0], bounds_2d[1, 1], 120)
    colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728']
    for c, offset in zip(colors, config.CURVE_OFFSETS):
        V_ent_curve = phase2_results['V_ent_center'] + offset
        if bounds_2d[0, 0] <= V_ent_curve <= bounds_2d[0, 1]:
            n_curve, sigma_curve = mapper.predict_curve(V_exit_curve, V_ent_curve)
            ax5.plot(V_exit_curve, n_curve, color=c, lw=2, label=f'$V_{{ENT}}$={V_ent_curve:.3f}V')
            ax5.fill_between(V_exit_curve, n_curve - sigma_curve, n_curve + sigma_curve,
                             color=c, alpha=0.15)
    ax5.axhline(1, color='green', ls='--', lw=2, alpha=0.7, label='n=1')
    ax5.set_xlabel('$V_{EXIT}$ (V)')
    ax5.set_ylabel('n')
    ax5.set_title('(e) Pumping curves at fixed $V_{ENT}$', fontsize=10)
    ax5.grid(alpha=0.25)
    ax5.legend(fontsize=8, loc='best')

    ax6 = fig.add_subplot(gs[1, 2])
    V_ent_best = phase2_results['V_ent_center']
    n_best_curve, _ = mapper.predict_curve(V_exit_curve, V_ent_best)
    dn_dV_best = np.gradient(n_best_curve, V_exit_curve)
    dI_dV_best_plot = config.target_current * dn_dV_best * 1e12
    ax6.plot(V_exit_curve, dI_dV_best_plot, 'r-', lw=2, label='GP line cut')

    half_tol = (bounds_2d[0, 1] - bounds_2d[0, 0]) / 20
    mask = np.abs(X_meas[:, 0] - V_ent_best) < half_tol
    if np.any(mask):
        X_local = X_meas[mask]
        y_local = n_meas[mask]
        if len(X_local) >= 3:
            order = np.argsort(X_local[:, 1])
            V_exit_local = X_local[order, 1]
            n_local = y_local[order]
            dn_dV_local = np.gradient(n_local, V_exit_local)
            dI_dV_local_plot = config.target_current * dn_dV_local * 1e12
            ax6.plot(V_exit_local, dI_dV_local_plot, 'ko', ms=4, alpha=0.6, label='Measured local')

    ax6.axhline(0, color='k', ls='--', lw=1)
    ax6.set_xlabel('$V_{EXIT}$ (V)')
    ax6.set_ylabel(r'$dI/dV_{EXIT}$ (pA/V)')
    ax6.set_title(f'(f) Line Cut at $V_{{ENT}}$={V_ent_best:.3f}V', fontsize=10)
    ax6.grid(alpha=0.25)
    ax6.legend(fontsize=8, loc='best')

    plt.suptitle(f'Pump Map Analysis [{mode_str}]', fontsize=15, fontweight='bold')
    plt.tight_layout(rect=[0, 0.00, 1, 0.96])

    config.output_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filepath = config.output_dir / f'phase5_pump_maps_{timestamp}.png'
    fig.savefig(filepath, dpi=170, bbox_inches='tight')
    plt.show()
    print(f'Saved: {filepath.resolve()}')
    return fig


print('Visualization functions defined')

In [ ]:
# Cell 9: Save Data Functions


def save_all_data_v5(
    phase1_results,
    phase2_attempts,
    phase2_selected_attempt,
    phase3_decision,
    config,
    phase4_results=None,
    phase5_results=None,
    phase5_decision=None,
):
    """
    Save all available outputs from v5 workflow.
    Includes Phase 2 attempt history and selected attempt.
    """
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    config.output_dir.mkdir(parents=True, exist_ok=True)

    saved_paths = []

    # Phase 1 traces
    for gate, tr in phase1_results['traces'].items():
        df_gate = pd.DataFrame({
            'gate': gate,
            'voltage_V': tr['voltage'],
            'current_A': tr['current_a'],
            'current_nA': tr['current_a'] * 1e9,
            'abs_current_A': tr['abs_current_a'],
        })
        p = config.output_dir / f'phase1_pinchoff_{gate}_{timestamp}.csv'
        df_gate.to_csv(p, index=False)
        saved_paths.append(p)
        print(f'Saved: {p}')

    # Phase 2 attempts
    for attempt in phase2_attempts:
        V_ent = attempt['V_ent_range']
        V_exit = attempt['V_exit_range']
        V_ENT, V_EXIT = np.meshgrid(V_ent, V_exit)

        df_attempt = pd.DataFrame({
            'attempt_idx': int(attempt['attempt_idx']),
            'V_ENT': V_ENT.ravel(),
            'V_EXIT': V_EXIT.ravel(),
            'V_P_fixed': attempt['V_p_fixed'],
            'I_A': attempt['current_map_a'].ravel(),
            'I_nA': attempt['current_map_na'].ravel(),
            'valid_mask_raw': attempt['valid_mask_raw'].astype(int).ravel(),
            'valid_mask_connected': attempt['valid_mask'].astype(int).ravel(),
        })

        p = config.output_dir / f'phase2_attempt_{attempt["attempt_idx"]}_{timestamp}.csv'
        df_attempt.to_csv(p, index=False)
        saved_paths.append(p)
        print(f'Saved: {p}')

    # Phase 4 optimization history (if completed)
    if phase4_results is not None:
        X4 = phase4_results['X_hist']
        df4 = pd.DataFrame({
            'iteration': np.arange(len(X4)),
            'V_ENT': X4[:, 0],
            'V_P': X4[:, 1],
            'V_EXIT': X4[:, 2],
            'n': phase4_results['n_hist'],
            'cost': phase4_results['y_hist'],
            'n_error': np.abs(phase4_results['n_hist'] - phase4_results.get('phase4_target_n', 1.0)),
        })
        p = config.output_dir / f'phase4_optimization_{timestamp}.csv'
        df4.to_csv(p, index=False)
        saved_paths.append(p)
        print(f'Saved: {p}')

    # Phase 5 mapping data (if completed)
    if phase5_results is not None:
        X5 = phase5_results['X_measured']
        df5 = pd.DataFrame({
            'point_idx': np.arange(len(X5)),
            'V_ENT': X5[:, 0],
            'V_EXIT': X5[:, 1],
            'V_P': phase5_results['V_p_fixed'],
            'n': phase5_results['n_measured'],
        })
        p = config.output_dir / f'phase5_mapping_{timestamp}.csv'
        df5.to_csv(p, index=False)
        saved_paths.append(p)
        print(f'Saved: {p}')

    # Summary
    selected_idx = np.nan if phase2_selected_attempt is None else int(phase2_selected_attempt + 1)
    selected_valid_points = np.nan
    if phase2_selected_attempt is not None and len(phase2_attempts) > 0:
        selected_valid_points = int(phase2_attempts[phase2_selected_attempt]['valid_points_connected'])

    summary_rows = [
        ('phase1_gain_A_per_V', phase1_results['gain_a_per_v']),
        ('phase1_pinch_V_ent', phase1_results['pinch_off_voltages']['V_ent']),
        ('phase1_pinch_V_p', phase1_results['pinch_off_voltages']['V_p']),
        ('phase1_pinch_V_exit', phase1_results['pinch_off_voltages']['V_exit']),
        ('phase1_measurements', phase1_results['total_measurements']),
        ('phase2_attempt_count', len(phase2_attempts)),
        ('phase2_selected_attempt', selected_idx),
        ('phase2_selected_valid_points', selected_valid_points),
        ('phase3_decision', phase3_decision),
        ('phase4_completed', phase4_results is not None),
        ('phase5_completed', phase5_results is not None),
        ('phase5_decision', phase5_decision if phase5_decision is not None else ''),
    ]

    if phase4_results is not None:
        summary_rows.extend([
            ('phase4_best_V_ent', phase4_results['best_V_ent']),
            ('phase4_best_V_p', phase4_results['best_V_p']),
            ('phase4_best_V_exit', phase4_results['best_V_exit']),
            ('phase4_best_n', phase4_results['best_n']),
            ('phase4_best_cost', phase4_results['best_cost']),
            ('phase4_total_measurements', phase4_results['total_measurements_including_refinement']),
        ])

    if phase5_results is not None:
        summary_rows.extend([
            ('phase5_measurements', len(phase5_results['X_measured'])),
            ('phase5_V_p_fixed', phase5_results['V_p_fixed']),
            ('phase5_V_ent_min', phase5_results['bounds_2d'][0, 0]),
            ('phase5_V_ent_max', phase5_results['bounds_2d'][0, 1]),
            ('phase5_V_exit_min', phase5_results['bounds_2d'][1, 0]),
            ('phase5_V_exit_max', phase5_results['bounds_2d'][1, 1]),
        ])

    summary = pd.DataFrame(summary_rows, columns=['Parameter', 'Value'])
    p_sum = config.output_dir / f'experiment_summary_v5_{timestamp}.csv'
    summary.to_csv(p_sum, index=False)
    saved_paths.append(p_sum)
    print(f'Saved: {p_sum}')

    return {
        'saved_paths': saved_paths,
        'summary_df': summary,
    }


print('v5 saving functions defined')

In [ ]:
# Cell 10: RUN Phase 1 + initial Phase 2

config = Config()
config.output_dir.mkdir(parents=True, exist_ok=True)

instr = InstrumentController(config)
print()
print(f'Mode: {instr.get_mode_string()}')

phase2_attempts = []
phase2_selected_attempt = None
phase3_decision = None
phase5_decision = None
phase4_results = None
phase5_results = None

PROCEED_TO_PHASE4 = False
PROCEED_TO_PHASE5 = False

phase1_results = run_phase1_pinchoff(instr, config)
display_phase1_results(phase1_results, config)
fig_phase1 = plot_phase1_pinchoff(phase1_results, config)

attempt_idx = 1
phase2_result = run_phase2_positive_current_mapping(instr, phase1_results, config, attempt_idx=attempt_idx)
phase2_attempts.append(phase2_result)
fig_phase2 = plot_phase2_positive_region(phase2_result, config)

In [ ]:
# Cell 11: PHASE 3 USER REVIEW (y/n/r)

print('='*70)
print('PHASE 3: REVIEW PHASE 2 RESULT')
print('='*70)

phase3_retry_count = 0

while True:
    latest = phase2_attempts[-1]
    print()
    print('Latest Phase 2 attempt summary:')
    print(f'  Attempt #:                  {latest["attempt_idx"]}')
    print(f'  Gain:                       {latest["gain_a_per_v"]:.0e} A/V')
    print(f'  Valid points (connected):   {latest["valid_points_connected"]}')
    print(f'  Valid bounds V_ENT:         [{latest["valid_bounds"][0,0]:+.4f}, {latest["valid_bounds"][0,1]:+.4f}] V')
    print(f'  Valid bounds V_EXIT:        [{latest["valid_bounds"][1,0]:+.4f}, {latest["valid_bounds"][1,1]:+.4f}] V')

    print()
    print('Decision options:')
    print('  y: approve Phase 2, manually switch amp gain dial to 1e-9 A/V, then continue to Phase 4')
    print('  n: stop now and save summary')
    print('  r: re-try Phase 2 with same settings')

    response = input('>>> Phase 3 decision [y/n/r]: ').strip().lower()

    if response in ['y', 'yes']:
        phase3_decision = 'y'
        phase2_selected_attempt = len(phase2_attempts) - 1
        PROCEED_TO_PHASE4 = True

        print()
        print('⚠️ Manual action required:')
        print('  1) Turn current amplifier gain dial to 1e-9 A/V')
        input('  2) After switching, press Enter to continue...')

        instr.set_current_amp_gain(config.GAIN_PHASE4_A_PER_V)
        print('✅ Gain sync complete. Proceeding to Phase 4.')
        break

    if response in ['n', 'no']:
        phase3_decision = 'n'
        PROCEED_TO_PHASE4 = False
        print()
        print('⏹️ Stopping after Phase 3 by user decision.')
        break

    if response in ['r', 'retry', 're-try']:
        phase3_decision = 'r'
        phase3_retry_count += 1
        next_attempt = len(phase2_attempts) + 1
        print()
        print(f'🔁 Re-running Phase 2 (attempt {next_attempt})...')
        phase2_result = run_phase2_positive_current_mapping(instr, phase1_results, config, attempt_idx=next_attempt)
        phase2_attempts.append(phase2_result)
        fig_phase2 = plot_phase2_positive_region(phase2_result, config)
        continue

    print('⚠️ Invalid input. Please enter y, n, or r.')

In [ ]:
# Cell 12: RUN PHASE 4 (Constrained n=1 optimization)

if not PROCEED_TO_PHASE4:
    print('='*70)
    print('⏹️ Phase 4 SKIPPED (Phase 3 decision was n)')
    print('='*70)
else:
    selected_phase2 = phase2_attempts[phase2_selected_attempt]
    print(f'Using Phase 2 attempt #{selected_phase2["attempt_idx"]} for constrained bounds.')

    phase4_results = run_phase4_optimization(instr, phase1_results, selected_phase2, config)
    display_phase4_results(phase4_results, config)
    fig_phase4 = plot_phase4_summary(phase4_results, instr, config)

In [ ]:
# Cell 13: PHASE 5 REVIEW + RUN PUMP MAP

if not PROCEED_TO_PHASE4:
    print('⏹️ Skipped - Phase 4 was not executed')
else:
    print('='*70)
    print('PHASE 5 GATE: REVIEW PHASE 4 RESULT')
    print('='*70)
    print(f'Phase 4 best n:     {phase4_results["best_n"]:.6f}')
    print(f'Phase 4 best |n-1|: {abs(phase4_results["best_n"] - phase4_results.get("phase4_target_n", 1.0)):.3e}')

    while True:
        resp = input('>>> Proceed to Phase 5 pump map? [y/n]: ').strip().lower()
        if resp in ['y', 'yes']:
            phase5_decision = 'y'
            PROCEED_TO_PHASE5 = True
            break
        if resp in ['n', 'no']:
            phase5_decision = 'n'
            PROCEED_TO_PHASE5 = False
            break
        print('⚠️ Invalid input. Please enter y or n.')

    if not PROCEED_TO_PHASE5:
        print()
        print('⏹️ Phase 5 skipped by user decision.')
    else:
        print()
        print('>>> Starting Phase 5 mapping...')
        phase5_results = run_phase5_mapping(instr, phase4_results, config)
        print('Generating Phase 5 6-panel pump map...')
        fig_phase5 = plot_phase2_maps(phase5_results, instr, config)

In [ ]:
# Cell 14: Save All Data (v5)

save_bundle = save_all_data_v5(
    phase1_results=phase1_results,
    phase2_attempts=phase2_attempts,
    phase2_selected_attempt=phase2_selected_attempt,
    phase3_decision=phase3_decision,
    config=config,
    phase4_results=phase4_results,
    phase5_results=phase5_results,
    phase5_decision=phase5_decision,
)

print()
print('--- Summary Preview ---')
print(save_bundle['summary_df'])

In [ ]:
# Cell 15: Final Summary & Cleanup

print()
print('='*70)
print('EXPERIMENT COMPLETE - FINAL SUMMARY (v5)')
print('='*70)
print(f'Mode: {instr.get_mode_string()}')

print()
print('--- Phase 1 ---')
print(f'  Gain:              {phase1_results["gain_a_per_v"]:.0e} A/V')
print(f'  Pinch V_ENT:       {phase1_results["pinch_off_voltages"]["V_ent"]:+.6f} V')
print(f'  Pinch V_P:         {phase1_results["pinch_off_voltages"]["V_p"]:+.6f} V')
print(f'  Pinch V_EXIT:      {phase1_results["pinch_off_voltages"]["V_exit"]:+.6f} V')
print(f'  Measurements:      {phase1_results["total_measurements"]}')

print()
print('--- Phase 2 ---')
print(f'  Attempts:          {len(phase2_attempts)}')
if phase2_selected_attempt is not None:
    selected = phase2_attempts[phase2_selected_attempt]
    print(f'  Selected attempt:  {selected["attempt_idx"]}')
    print(f'  Valid points:      {selected["valid_points_connected"]}')
else:
    print('  Selected attempt:  None')

print()
print('--- Phase 3 ---')
print(f'  Decision:          {phase3_decision}')

if phase4_results is not None:
    print()
    print('--- Phase 4 ---')
    print(f'  Gain:              {instr.current_amp_gain:.0e} A/V')
    print(f'  Best n:            {phase4_results["best_n"]:.6f}')
    print(f'  Best |n-1|:        {abs(phase4_results["best_n"] - phase4_results.get("phase4_target_n", 1.0)):.3e}')

if phase5_results is not None:
    print()
    print('--- Phase 5 ---')
    print(f'  Decision:          {phase5_decision}')
    print(f'  Measurements:      {len(phase5_results["X_measured"])}')

print()
print('Output directory:')
print(f'  {config.output_dir.resolve()}')
for f in sorted(config.output_dir.glob('*')):
    print(f'    {f.name}')

print('='*70)

instr.close()